Here I download data from API ISTAT and save it locally so I don't have to call the API everytime

In [2]:
import os
import requests
import pandas as pd

In [3]:
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
file_path = "../data/raw/istat_road_accidents.csv"

In [4]:
if os.path.exists(file_path):
    print("file already downloaded")
else: 
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print("file downloaded")
    else:
        print("Error:", response.status_code)

file downloaded


First look at the data

In [4]:
df = pd.read_csv(file_path)
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.shape


(573552, 16)

The column REF_AREA don't have anymore the initial zeros shown in the csv file because it's read as a number.

In [8]:
df = pd.read_csv(file_path, dtype={"REF_AREA": str})
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,001001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,001001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,001001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,001001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,001001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.dtypes

DATAFLOW                str
FREQ                    str
REF_AREA                str
DATA_TYPE               str
RESULT                  str
TIME_PERIOD           int64
OBS_VALUE             int64
OBS_STATUS          float64
NOTE_DS             float64
NOTE_REF_AREA       float64
NOTE_DATA_TYPE      float64
NOTE_RESULT         float64
NOTE_TIME_PERIOD    float64
BASE_PER            float64
UNIT_MEAS           float64
UNIT_MULT           float64
dtype: object

Controlling the columns

In [10]:
df.isnull().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          573552
NOTE_DS             573552
NOTE_REF_AREA       573552
NOTE_DATA_TYPE      573552
NOTE_RESULT         573552
NOTE_TIME_PERIOD    573552
BASE_PER            573552
UNIT_MEAS           573552
UNIT_MULT           573552
dtype: int64

In [11]:
df["DATA_TYPE"].value_counts()

DATA_TYPE
KILLINJ    382368
ROADACC    191184
Name: count, dtype: int64

In [12]:
df["RESULT"].value_counts()

RESULT
F    191184
M    191184
9    191184
Name: count, dtype: int64

In [13]:
df["TIME_PERIOD"].unique()

array([2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,
       2023, 2024])

I noticed that column KILLINJ has twice the number of rows as column ROADACC and F,M,9 have the same amount of rows. I want to check the combination between DATA_TYPE and RESULT.

In [15]:
df.groupby(["DATA_TYPE", "RESULT"]).size()

DATA_TYPE  RESULT
KILLINJ    F         191184
           M         191184
ROADACC    9         191184
dtype: int64

So 9 refers only to ROADACC

In [17]:
df[(df["REF_AREA"] == "001001") & (df["TIME_PERIOD"] == 2023)]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
22,IT1:41_983(1.0),A,001001,KILLINJ,F,2023,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,IT1:41_983(1.0),A,001001,KILLINJ,M,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,IT1:41_983(1.0),A,001001,ROADACC,9,2023,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df[(df["REF_AREA"] == "001002") & (df["TIME_PERIOD"] == 2023)]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
94,IT1:41_983(1.0),A,001002,KILLINJ,F,2023,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118,IT1:41_983(1.0),A,001002,KILLINJ,M,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,IT1:41_983(1.0),A,001002,ROADACC,9,2023,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


I see that for each municipality it gives alwas three rows such as KILLINJ F, KILLINJ M, ROADACC.
So KILLINJ refers to kills (M=morti), and injuries (F=feriti); ROADACC is road accidents and I think 9 does not give any information but it can be considered as null or without a meaning because it refers only to road accidents, not a person.
I notice also that there are more injuries than kills and also than accidents.

Now I want to see if F AND M refers to "feriti" and "morti": if the sum of M and F rows is equal to the official ISTAT numbers. If it's half of the people it refers to "maschi". I want also to check ROADACC.

In [19]:
df_2023 = df[df["TIME_PERIOD"] == 2023]
df_2023.groupby(["DATA_TYPE", "RESULT"])["OBS_VALUE"].sum()

DATA_TYPE  RESULT
KILLINJ    F         224634
           M           3039
ROADACC    9         166525
Name: OBS_VALUE, dtype: int64

The 2023 totals match the official ISTAT data so my interpretation is confirmed.